# Análise Estatística
## Análise de Dados de Compliance Pública - Tese de MBA

**Objetivo:** Realizar análise estatística rigorosa:
- Análise de correlação
- Testes de hipóteses
- Regressão linear múltipla
- Diagnósticos e validação do modelo

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


## Passo a passo (estilo aula)

1. Pacotes e configuração do ambiente
2. Reprodutibilidade
3. Carregamento dos dados
4. Blocos de análise
5. Resumo e interpretação


# Pacotes


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from scipy.stats import shapiro, normaltest, jarque_bera

# Try local loader first, fallback to S3 if available
from src.analysis.local_data_loader import LocalGoldDataLoader as GoldDataLoader

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)
pd.set_option('display.float_format', '{:.4f}'.format)


In [ ]:
import matplotlib as mpl
mpl.rcParams['axes.formatter.useoffset'] = False
mpl.rcParams['axes.formatter.limits'] = (-99, 99)


# Reprodutibilidade


In [ ]:
import os
import random

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Semente de reprodutibilidade fixada em {SEED}")


In [ ]:
import json as _json

_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}

S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))
AWS_PROFILE = os.environ.get('AWS_PROFILE', _rtcfg.get('aws', {}).get('profile', None))


## 1. Carregar Dados

In [ ]:
from src.analysis.pt_br_loader import GoldDataLoaderPtBr as GoldDataLoader
loader = GoldDataLoader()

# --- Análise em nível municipal (N ~= 5.570) -----------------------------
# Agora carregamos o dataset municipal `analise_compliance_municipio` em vez
# do dataset estadual `analise_compliance` (N=27). A granularidade municipal
# garante poder estatístico adequado para correlação, OLS e ML abaixo.
#
# Para manter compatibilidade com o restante do notebook:
#   1. Renomeamos colunas do nível municipal para os antigos nomes estaduais
#      (ex.: `populacao_2022` -> `populacao`), preservando células a jusante.
#   2. Recriamos as dummies regionais humanamente legíveis (is_norte, is_nordeste,
#      is_sudeste, is_sul, is_centro_oeste) com os mesmos nomes do dataset
#      estadual.
#   3. Adicionamos dummies de estado (is_state_<código-IBGE>) como features extras.
df = loader.load_dataset('analise_compliance_municipio')
df = df.rename(columns={
    'populacao_2022': 'populacao',
    'taxa_alfabetizacao_2022': 'taxa_alfabetizacao_media',
    'renda_media_2022': 'renda_media',
})

REGION_NAME_TO_DUMMY = {
    'Norte': 'is_norte',
    'Nordeste': 'is_nordeste',
    'Sudeste': 'is_sudeste',
    'Sul': 'is_sul',
    'Centro-Oeste': 'is_centro_oeste',
}
for _rname, _col in REGION_NAME_TO_DUMMY.items():
    df[_col] = (df['nome_regiao'] == _rname).astype('Int64')
REGION_DUMMY_COLS = list(REGION_NAME_TO_DUMMY.values())

state_dummies = pd.get_dummies(df['codigo_estado'], prefix='is_state').astype('Int64')
df = pd.concat([df, state_dummies], axis=1)
STATE_DUMMY_COLS = list(state_dummies.columns)

print(f"Carregadas {len(df):,} observações (municípios em {df['codigo_estado'].nunique()} estados)")
print(f"Dummies regionais: {REGION_DUMMY_COLS}")
print(f"Dummies de estado: {len(STATE_DUMMY_COLS)} colunas (primeira: {STATE_DUMMY_COLS[0]}, última: {STATE_DUMMY_COLS[-1]})")
df.head()

## 2. Análise de Correlação

### 2.1 Pearson Matriz de Correlação

In [ ]:
# Variáveis analíticas principais (nível municipal). Removemos
# `num_municipios` pois é constante nessa granularidade. Adicionamos
# `log_total_transferencias` quando disponível para expor o sinal do lado
# das transferências (central para a pergunta do TCC).
key_vars = ['sancoes_por_100k', 'taxa_alfabetizacao_media', 'renda_media',
            'log_populacao', 'log_renda']
if 'log_total_transferencias' in df.columns:
    key_vars.append('log_total_transferencias')

# Cast para float puro -- tipos Int64/Float64 nullable com NaN podem dar
# problema no heatmap do seaborn.
corr_matrix = df[key_vars].astype('Float64').astype(float).corr(method='pearson')

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlação de Pearson (nível municipal, N=5.570)',
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\nCorrelações com Sanções por 100k:")
print("=" * 60)
sanctions_corr = corr_matrix['sancoes_por_100k'].sort_values(ascending=False)
print(sanctions_corr)

### 2.2 Teste de Significância Estatística

In [ ]:
from scipy.stats import pearsonr

def correlation_test(x, y, var_names):
    """Calcula correlação de Pearson com valor-p."""
    clean_data = pd.DataFrame({'x': x, 'y': y}).dropna()
    if len(clean_data) < 3:
        return None, None
    r, p = pearsonr(clean_data['x'].astype(float), clean_data['y'].astype(float))
    return r, p

results = []
target = df['sancoes_por_100k']

_test_vars = ['taxa_alfabetizacao_media', 'renda_media', 'log_renda', 'log_populacao']
if 'log_total_transferencias' in df.columns:
    _test_vars.append('log_total_transferencias')

for var in _test_vars:
    r, p = correlation_test(target, df[var], (var, 'sancoes_por_100k'))
    if r is not None:
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        results.append({
            'Variável': var,
            'Correlação (r)': f"{r:.4f}",
            'P-valor': f"{p:.4f}",
            'Significância': sig,
        })

print("Testes de correlação de Pearson vs. sancoes_por_100k (N=5.570)")
print("=" * 60)
print(pd.DataFrame(results).to_string(index=False))

### 2.3 Gráficos de Dispersão com Linhas de Regressão

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

vars_to_plot = [
    ('avg_literacy_rate', 'Taxa de Alfabetização (%)', axes[0, 0]),
    ('avg_income', 'Renda Média (BRL)', axes[0, 1]),
    ('log_income', 'Log(Renda)', axes[1, 0]),
    ('log_population', 'Log(População)', axes[1, 1])
]

for var, label, ax in vars_to_plot:
    ax.scatter(df[var], df['sanctions_per_100k'], alpha=0.6, s=100)
    
    z = np.polyfit(df[var].dropna(), df['sanctions_per_100k'][df[var].notna()], 1)
    p = np.poly1d(z)
    ax.plot(df[var].sort_values(), p(df[var].sort_values()), "r--", alpha=0.8, linewidth=2)
    
    r, pval = correlation_test(df[var], df['sanctions_per_100k'], (var, 'sanctions'))
    ax.text(0.05, 0.95, f'r = {r:.3f}\np = {pval:.4f}', 
            transform=ax.transAxes, fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel(label, fontsize=12)
    ax.set_ylabel('Sanções por 100 mil', fontsize=12)
    ax.set_title(f'Sanções vs {label}', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Testes de Hipótese

### 3.1 Diferenças Regionais (ANOVA)

In [ ]:
from scipy.stats import f_oneway

regions = df['region_name'].unique()
groups = [df[df['region_name'] == region]['sanctions_per_100k'].dropna() for region in regions]

f_stat, p_value = f_oneway(*groups)

print("ANOVA One-Way: Diferenças Regionais em Sanções por 100 mil")
print("=" * 70)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"\nResultado: {'Significativo' if p_value < 0.05 else 'Não significativo'} diferenças regionais")
print("\nMédias Regionais:")
print(df.groupby('region_name')['sanctions_per_100k'].agg(['mean', 'std', 'count']).round(2))


In [ ]:
plt.figure(figsize=(12, 6))
df.boxplot(column='sanctions_per_100k', by='region_name', ax=plt.gca())
plt.title('Sanções por 100 mil por Região', fontsize=14, fontweight='bold')
plt.suptitle('')
plt.xlabel('Região', fontsize=12)
plt.ylabel('Sanções por 100 mil', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 3.2 Testes Post-hoc (Tukey HSD)

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(endog=df['sanctions_per_100k'], 
                          groups=df['region_name'], 
                          alpha=0.05)

print("Teste Post-hoc Tukey HSD")
print("=" * 80)
print(tukey)

tukey.plot_simultaneous()
plt.tight_layout()
plt.show()


## 4. Regressão Linear Múltipla

### 4.1 Especificação do Modelo

In [ ]:
y = df['sancoes_por_100k']

# Removemos `is_sudeste` como região de referência para evitar a armadilha
# das dummies (dummy variable trap).
X_vars = ['log_renda', 'taxa_alfabetizacao_media', 'log_populacao',
          'is_norte', 'is_nordeste', 'is_sul', 'is_centro_oeste']
if 'log_total_transferencias' in df.columns:
    X_vars.append('log_total_transferencias')

# Descarta linhas com NaN em qualquer feature (o nível municipal tem algumas
# colunas nullable) e converte para float puro para o statsmodels.
_mask = df[X_vars + ['sancoes_por_100k']].notna().all(axis=1)
X = df.loc[_mask, X_vars].copy().astype(float)
y = df.loc[_mask, 'sancoes_por_100k'].astype(float)
X = sm.add_constant(X)

print("Especificação do Modelo:")
print("=" * 70)
print(f"Variável Dependente: sancoes_por_100k (nível municipal)")
print(f"Variáveis Independentes: {X_vars}")
print(f"Região de referência (omitida para evitar armadilha das dummies): Sudeste")
print(f"\nTamanho da amostra: {len(X):,} municípios (linhas com NaN descartadas)")
print(f"Número de preditores: {len(X_vars)}")

### 4.2 Regressão MQO (Mínimos Quadrados Ordinários)

In [ ]:
model = sm.OLS(y, X).fit()

print(model.summary())


### 4.3 Interpretação dos Coeficientes

In [ ]:
coef_df = pd.DataFrame({
    'Variável': model.params.index,
    'Coeficiente': model.params.values,
    'Erro Padrão': model.bse.values,
    'Estatística-t': model.tvalues.values,
    'Valor-p': model.pvalues.values,
    'IC Inferior': model.conf_int()[0].values,
    'IC Superior': model.conf_int()[1].values
})

coef_df['Significativo'] = coef_df['Valor-p'].apply(
    lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
)

print("\nCoeficientes de Regressão")
print("=" * 100)
display(coef_df.round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

coef_plot = coef_df[coef_df['Variável'] != 'const'].copy()
coef_plot = coef_plot.sort_values('Coeficiente')

colors = ['red' if p < 0.05 else 'gray' for p in coef_plot['Valor-p']]

ax.barh(coef_plot['Variável'], coef_plot['Coeficiente'], color=colors, alpha=0.7)
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Valor do Coeficiente', fontsize=12)
ax.set_title('Coeficientes de Regressão (Vermelho = Significativo em p<0.05)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


## 5. Diagnósticos do Modelo

### 5.1 Análise de Resíduos

In [ ]:
residuals = model.resid
fitted = model.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].scatter(fitted, residuals, alpha=0.6)
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Valores Ajustados')
axes[0, 0].set_ylabel('Resíduos')
axes[0, 0].set_title('Resíduos vs Ajustados', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Gráfico Q-Q', fontweight='bold')

axes[1, 0].hist(residuals, bins=15, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Resíduos')
axes[1, 0].set_ylabel('Frequência')
axes[1, 0].set_title('Histograma dos Resíduos', fontweight='bold')
axes[1, 0].axvline(x=0, color='r', linestyle='--')

standardized_resid = residuals / np.std(residuals)
axes[1, 1].scatter(fitted, np.sqrt(np.abs(standardized_resid)), alpha=0.6)
axes[1, 1].set_xlabel('Valores Ajustados')
axes[1, 1].set_ylabel('√|Resíduos Padronizados|')
axes[1, 1].set_title('Gráfico Escala-Localização', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 5.2 Testes de Normalidade

In [ ]:
shapiro_stat, shapiro_p = shapiro(residuals)
jb_stat, jb_p = jarque_bera(residuals)

print("Testes de Normalidade dos Resíduos")
print("=" * 70)
print(f"Shapiro-Wilk Test:")
print(f"  Estatística: {shapiro_stat:.4f}")
print(f"  P-value: {shapiro_p:.4f}")
print(f"  Resultado: {'Rejeitar normalidade' if shapiro_p < 0.05 else 'Não pode rejeitar normalidade'}\n")

print(f"Jarque-Bera Test:")
print(f"  Estatística: {jb_stat:.4f}")
print(f"  P-value: {jb_p:.4f}")
print(f"  Resultado: {'Rejeitar normalidade' if jb_p < 0.05 else 'Não pode rejeitar normalidade'}")


### 5.3 Testes de Heterocedasticidade

In [ ]:
bp_stat, bp_p, _, _ = het_breuschpagan(residuals, X)

print("Testes de Heterocedasticidade")
print("=" * 70)
print(f"Breusch-Pagan Test:")
print(f"  Estatística: {bp_stat:.4f}")
print(f"  P-value: {bp_p:.4f}")
print(f"  Resultado: {'Heterocedasticidade detectada' if bp_p < 0.05 else 'Homocedasticidade (variância constante)'}")


### 5.4 Multicolinearidade (VIF)

In [ ]:
X_no_const = X.drop('const', axis=1)

vif_data = pd.DataFrame()
vif_data['Variável'] = X_no_const.columns
vif_data['VIF'] = [variance_inflation_factor(X_no_const.values, i) 
                   for i in range(X_no_const.shape[1])]

vif_data['Multicolinearidade'] = vif_data['VIF'].apply(
    lambda x: 'Alto (>10)' if x > 10 else 'Moderado (5-10)' if x > 5 else 'Baixo (<5)'
)

print("Análise do Fator de Inflação da Variância (VIF)")
print("=" * 70)
print("Regra geral: VIF > 10 indica alta multicolinearidade\n")
display(vif_data.sort_values('VIF', ascending=False))


### 5.5 Observações Influentes

In [ ]:
from statsmodels.stats.outliers_influence import OLSInfluence

influence = OLSInfluence(model)
cooks_d = influence.cooks_distance[0]

fig, ax = plt.subplots(figsize=(14, 6))
ax.stem(range(len(cooks_d)), cooks_d, markerfmt=',')
ax.axhline(y=4/len(X), color='r', linestyle='--', label='Limiar (4/n)')
ax.set_xlabel('Índice da Observação')
ax.set_ylabel("Cook's Distance")
ax.set_title("Cook's Distance - Observações Influentes", fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

threshold = 4 / len(X)
influential = df[cooks_d > threshold][['state_name', 'sanctions_per_100k']]
if len(influential) > 0:
    print(f"\nObservações influentes (D de Cook > {threshold:.4f}):")
    print(influential)
else:
    print("\nNenhuma observação altamente influente detectada.")


## 6. Comparação de Modelos

In [ ]:
# Baseline de duas variáveis para comparação (mesmo tamanho de amostra).
_simple_vars = ['log_renda', 'taxa_alfabetizacao_media']
X_simple = sm.add_constant(df.loc[_mask, _simple_vars].astype(float))
model_simple = sm.OLS(y, X_simple).fit()

model_full = model

comparison = pd.DataFrame({
    'Modelo': [f'Simples ({len(_simple_vars)} vars)', f'Completo ({len(X_vars)} vars + dummies de região)'],
    'N': [int(model_simple.nobs), int(model_full.nobs)],
    'R-quadrado': [model_simple.rsquared, model_full.rsquared],
    'R-quadrado ajustado': [model_simple.rsquared_adj, model_full.rsquared_adj],
    'AIC': [model_simple.aic, model_full.aic],
    'BIC': [model_simple.bic, model_full.bic],
    'F': [model_simple.fvalue, model_full.fvalue],
    'Prob (F)': [model_simple.f_pvalue, model_full.f_pvalue],
})

print("Comparação de Modelos (nível municipal)")
print("=" * 80)
display(comparison.round(4))

print("\nObs.: AIC/BIC menor indica melhor ajuste")

## 7. Resumo das Descobertas

### 7.1 Análise de Correlação
- **Renda média** é o preditor mais forte de sanções por 100 mil (r = 0,74, p < 0,001).
- **Renda log-transformada** também apresenta correlação forte (r = 0,63).
- **Taxa de alfabetização** tem associação positiva moderada (r = 0,47).
- Tamanho da população e número de municípios mostram associações negativas fracas.

### 7.2 Diferenças Regionais
- **ANOVA one-way** não encontrou diferenças estatisticamente significativas entre regiões (F = 1,70; p = 0,186).
- **Tukey HSD** confirma que nenhuma comparação regional par a par é significativa em α = 0,05.
- Centro-Oeste tem a maior média (25,66), mas também a maior variância (dp = 26,74), influenciada pelo outlier do Distrito Federal.

### 7.3 Resultados da Regressão
- **Regressão OLS** alcança R² = 0,835 (R² Ajustado = 0,775), explicando a maior parte da variância nas taxas de sanções.
- **Log da renda** é o preditor significativo mais forte (β = 49,75; p < 0,001): um aumento de 1% na renda média está associado a ~0,50 sanções adicionais por 100 mil.
- **Dummies regionais** (Norte: β = 20,94, p = 0,003; Nordeste: β = 22,97, p = 0,009) são significativas, indicando que, controlando pela renda, essas regiões têm taxas de sanções acima do esperado.
- **Taxa de alfabetização** não é significativa quando a renda já está no modelo (p = 0,988), sugerindo que sua correlação bivariada é mediada pela renda.

### 7.4 Diagnósticos do Modelo
- **Normalidade**: Shapiro-Wilk (p = 0,094) e Jarque-Bera (p = 0,340) — resíduos são aproximadamente normais.
- **Homocedasticidade**: teste de Breusch-Pagan (p = 0,299) — sem evidência de heterocedasticidade.
- **Observações influentes**: Distrito Federal (D de Cook > limiar) é o ponto mais influente, junto com Tocantins e Paraná.

### 7.5 Limitações
- Amostra pequena (n = 27 estados) limita o poder estatístico e o número de preditores.
- Desenho transversal não permite estabelecer causalidade — renda mais alta pode correlacionar com maior capacidade institucional de detectar e registrar sanções, não com mais irregularidades de fato.
- O Distrito Federal é um outlier estrutural (capital federal com governança única) que influencia fortemente os resultados.
